# The optimiser's LCA, read in one place

Every inventory the Data Centre Optimiser scores lives in `optimiser_lca/`, one
module per LCA, each in the same four sections: source and scope (the module
docstring), parameters, processes, activities. This notebook *reads* them: for
each inventory it prints every process's search neighbourhood with the chosen
row marked and its reason, then one table per activity with the ecoinvent
activity each exchange resolves to.

Nothing here writes to Brightway. To (re)build the databases:

```
python -m optimiser_lca.build            # all six, in dependency order
python -m optimiser_lca.build fuel_cell  # one of them
python optimisation/coefficients.py      # refresh the optimiser's coefficient caches
```

To change an inventory, edit its module (a query and pick to choose a
different ecoinvent process, an amount, a parameter), re-run its section here
to see the result, then rebuild. The build resolves every process by its
pinned exact identity and refuses to proceed if that identity is no longer
among the search hits for its query, so a ranking shift can never swap a
process silently.

| section | module | database | what the optimiser reads from it |
|---|---|---|---|
| Grid | `optimiser_lca/grid.py` | (scores, not a database) | `grid_build.py` — the half-hourly series |
| Hydrogen | `optimiser_lca/hydrogen.py` | `hydrogen foreground` | `electrolysers.py`, `h2_prices.py` |
| Data centre | `optimiser_lca/data_centre.py` | `data centre foreground` | `baseline_grid.py` |
| Fuel cell | `optimiser_lca/fuel_cell.py` | `fuel cell foreground` | `fuel_cell.py` |
| H2 tank | `optimiser_lca/h2_tank.py` | `hydrogen storage foreground` | `h2_storage.py` |
| Step-up transformers | `optimiser_lca/stepup_transformers.py` | `transformer foreground` | `transformers.py` |
| Utility PV | `optimiser_lca/upv.py` | `solar foreground` | `renewables.py` |

In [ ]:
import pandas as pd
import dc_config as cfg
import lca_helpers as H
from optimiser_lca import common, registry, build
from optimiser_lca import grid, hydrogen, data_centre, fuel_cell, h2_tank, stepup_transformers, upv

cfg.print_dashboard()
print()
ei, bio, fg_db, method = H.setup_brightway()
pd.set_option("display.width", 200)

## Grid

`optimiser_lca/grid.py` — the ecoinvent source per Carbon Intensity API fuel,
the wind and imports splits, the loss factor, the closed-form score, and the
network delivery terms bought green hydrogen pays. The id printed is the one
every grid CSV on disk carries; change a source and every series goes stale
(`python -m optimisation.grid_build --rescore` brings them back).

In [ ]:
print(grid.__doc__)
grid.show(candidates=True)

## Hydrogen — Hermesmann et al.

Six routes: SMR, SMR-CCS (with its CCS waste treatment), methane pyrolysis
MP-E, and alkaline / PEM / SOEC electrolysis as a 1 MW capital good plus an
operation activity each. `optimiser_lca/hydrogen.py` holds the source notes,
every process (with its search and the reason for the pick), and the amounts.

In [ ]:
print(hydrogen.__doc__)
_ = common.show(hydrogen.inventory())

## Data centre — Zhang et al. 2025

In [ ]:
print(data_centre.__doc__)
_ = common.show(data_centre.inventory())

## PEM fuel cell — Babatunde et al. 2024, and 1 kWh of electricity from it

The stack's amounts are shown as printed in Table S1 and the two scale
factors applied once (the `x0.939…` on the activity header). The operation
activity's technician-travel exchange is read out of ecoinvent's maintenance
dataset when the inventory is resolved, so it appears below only after
`show()` has run.

In [ ]:
print(fuel_cell.__doc__)
print(fuel_cell.describe())
_ = common.show(fuel_cell.inventory())

### Validation against the paper

Reproduces Babatunde et al.'s Tables 3, 4 and 5 on ReCiPe 2016 (H) from the
stack activity as written, then sanity-checks the operation activity: where
its per-kWh burden comes from, how it compares with ecoinvent's own PEMFC
operation once the domestic-CHP artefacts are stripped, and how hard the
efficiency assumption pushes. Needs the databases built.

In [ ]:
from optimiser_lca import validate
_ = validate.fc_stack_vs_paper()
_ = validate.fc_operation_sanity()
_ = validate.data_centre_vs_paper()

## Hydrogen storage tank — Gandiglio et al. 2022

In [ ]:
print(h2_tank.__doc__)
print(h2_tank.describe())
_ = common.show(h2_tank.inventory())

## Generation step-up transformers — Leng et al. 2026 (wind), IEA PVPS 2026 (solar)

In [ ]:
print(stepup_transformers.__doc__)
print(stepup_transformers.check())
print(stepup_transformers.describe())
_ = common.show(stepup_transformers.inventory())

## Utility-scale PV — Smith et al. (NREL 2024), fixed tilt, rescored on apos

Fourteen activities: Smith's eleven nested unit processes, the IEA PVPS
fixed-tilt mounting, the plant per MWp, and thirty years of O&M per MWp. The
search neighbourhoods are skipped for this one (135 processes, nearly all
resolved from Smith's own ecoinvent-native names); the tables show what each
row resolved to.

In [ ]:
print(upv.__doc__)
print(upv.check())
print(upv.describe())
_ = common.show(upv.inventory(), candidates=False)

## Build

Run from the shell (`python -m optimiser_lca.build`) or here. Deletes and
rewrites each database whole, in dependency order. Afterwards refresh the
coefficient caches with `python optimisation/coefficients.py`.

In [ ]:
# build.main([])            # all six
# build.main(["fuel_cell"]) # just one